# Path-Dependent Option Pricing

## Beyond Vanilla Payoffs

In Chapters 4 and 5, the payoff depended only on the **terminal** asset price $S_T$ (or on the price at some optimal stopping time). Many real-world derivatives depend on the **entire path** $\{S_t\}_{0 \leq t \leq T}$. These are called **path-dependent** or **exotic** options.

We will price three important types:

| Option | Payoff depends on | Example payoff (call) |
|--------|-------------------|----------------------|
| **Asian** | Average price $\bar{S} = \frac{1}{M}\sum_{k} S_{t_k}$ | $\max(\bar{S} - K, 0)$ |
| **Barrier** (knock-out) | Whether $S_t$ hits a barrier $B$ | $\max(S_T - K, 0) \cdot \mathbf{1}_{\max_t S_t < B}$ |
| **Lookback** | Running max/min of $S_t$ | $S_{\max} - S_T$ (floating strike) |

Since these payoffs depend on the full trajectory, **closed-form solutions are rare or complex**. Monte Carlo simulation handles them naturally — we just compute the payoff from the simulated path.

The Feynman-Kac connection still holds: we are computing $V = e^{-rT}\,\mathbb{E}[g(\{S_t\})]$, where $g$ is now a functional of the path rather than a function of $S_T$ alone.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, log, sqrt, exp
%matplotlib inline

def bs_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def bs_call(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S * bs_cdf(d1) - K * exp(-r*T) * bs_cdf(d2)

def bs_put(S, K, r, T, sigma):
    d1 = (log(S/K) + (r + sigma**2/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return -S * bs_cdf(-d1) + K * exp(-r*T) * bs_cdf(-d2)

def simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths):
    """Simulate GBM paths. Returns shape (n_steps+1, n_paths)."""
    dt = T / n_steps
    Z = np.random.randn(n_steps, n_paths)
    log_inc = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_S = np.log(S0) + np.cumsum(log_inc, axis=0)
    return np.exp(np.vstack([np.full((1, n_paths), np.log(S0)), log_S]))

In [ ]:
# Simulate paths
np.random.seed(42)
S0, K, r, T, sigma = 100, 100, 0.05, 1.0, 0.2
n_steps, n_paths = 252, 500000  # daily steps for 1 year

S = simulate_gbm_paths(S0, r, sigma, T, n_steps, n_paths)
t_grid = np.linspace(0, T, n_steps + 1)

# Plot a few sample paths with key path statistics highlighted
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(5):
    ax.plot(t_grid, S[:, i], alpha=0.6, linewidth=0.8)
    # Mark the running max
    ax.scatter(t_grid[np.argmax(S[:, i])], np.max(S[:, i]), marker='^', s=50, zorder=5)

ax.axhline(K, color='k', linestyle='--', alpha=0.3, label=f'K = {K}')
ax.set_xlabel('Time')
ax.set_ylabel('$S_t$')
ax.set_title('Sample Paths (triangles = path maximum)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Simulated {n_paths:,} paths with {n_steps} time steps")

## 1. Asian Options

An **Asian option** payoff depends on the **average** asset price over the option's life, rather than the terminal price. This averaging reduces volatility and makes the option cheaper than its vanilla counterpart.

**Fixed-strike Asian call**: $\text{payoff} = \max(\bar{S} - K, 0)$, where $\bar{S} = \frac{1}{M}\sum_{k=0}^{M} S_{t_k}$

**Fixed-strike Asian put**: $\text{payoff} = \max(K - \bar{S}, 0)$

There is no simple closed-form price. The difficulty is that the sum of lognormals is not lognormal. However, MC handles it trivially.

In [ ]:
# Asian option pricing
S_avg = np.mean(S, axis=0)  # arithmetic average over all time steps
S_T = S[-1]

asian_call_payoff = np.maximum(S_avg - K, 0)
asian_put_payoff = np.maximum(K - S_avg, 0)

asian_call = np.mean(np.exp(-r*T) * asian_call_payoff)
asian_put = np.mean(np.exp(-r*T) * asian_put_payoff)
asian_call_err = np.std(np.exp(-r*T) * asian_call_payoff) / np.sqrt(n_paths)
asian_put_err = np.std(np.exp(-r*T) * asian_put_payoff) / np.sqrt(n_paths)

# Compare with vanilla European
euro_call = bs_call(S0, K, r, T, sigma)
euro_put = bs_put(S0, K, r, T, sigma)

print("=== Asian vs European Options ===")
print(f"{'':20s} {'Asian (MC)':>14s} {'European (BS)':>14s} {'Ratio':>8s}")
print(f"{'Call':20s} {asian_call:14.4f} {euro_call:14.4f} {asian_call/euro_call:8.2%}")
print(f"{'Put':20s} {asian_put:14.4f} {euro_put:14.4f} {asian_put/euro_put:8.2%}")
print(f"\nAsian options are cheaper because averaging reduces effective volatility.")

In [ ]:
# Visualize: distribution of average price vs terminal price
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(S_T, bins=200, density=True, alpha=0.5, label='$S_T$ (terminal)', range=(50, 180))
ax.hist(S_avg, bins=200, density=True, alpha=0.5, label=r'$\bar{S}$ (average)', range=(50, 180))
ax.axvline(K, color='k', linestyle='--', alpha=0.5, label=f'K = {K}')
ax.set_xlabel('Price')
ax.set_ylabel('Density')
ax.set_title(r'Distribution of $S_T$ vs $\bar{S}$: averaging narrows the distribution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Barrier Options

A **barrier option** is activated or deactivated when the asset price hits a specified barrier level $B$. There are four types:

| Type | Condition | Behavior |
|------|-----------|----------|
| **Up-and-out** | $\max_t S_t \geq B$ | Knocked out (worthless) |
| **Up-and-in** | $\max_t S_t \geq B$ | Knocked in (becomes active) |
| **Down-and-out** | $\min_t S_t \leq B$ | Knocked out |
| **Down-and-in** | $\min_t S_t \leq B$ | Knocked in |

A useful **parity relation**: knock-in + knock-out = vanilla. So pricing one gives the other for free.

Barrier options are cheaper than vanilla options because there's a chance they get knocked out (or never knock in). They are popular for hedging with reduced premium.

In [ ]:
# Barrier option pricing
B_up = 130    # upper barrier
B_down = 80   # lower barrier

S_max = np.max(S, axis=0)   # running max per path
S_min = np.min(S, axis=0)   # running min per path

vanilla_call_payoff = np.maximum(S_T - K, 0)
vanilla_put_payoff = np.maximum(K - S_T, 0)

# Up-and-out call: knocked out if S ever reaches B_up
uo_call_payoff = vanilla_call_payoff * (S_max < B_up)
# Up-and-in call: active only if S reaches B_up
ui_call_payoff = vanilla_call_payoff * (S_max >= B_up)

# Down-and-out put: knocked out if S ever drops to B_down
do_put_payoff = vanilla_put_payoff * (S_min > B_down)
# Down-and-in put: active only if S drops to B_down
di_put_payoff = vanilla_put_payoff * (S_min <= B_down)

def mc_stats(payoffs):
    disc = np.exp(-r*T) * payoffs
    return np.mean(disc), np.std(disc) / np.sqrt(len(disc))

print(f"=== Barrier Options (B_up={B_up}, B_down={B_down}) ===")
print(f"{'':25s} {'Price':>10s} {'Std Err':>10s}")
print(f"{'Vanilla call':25s} {mc_stats(vanilla_call_payoff)[0]:10.4f} {mc_stats(vanilla_call_payoff)[1]:10.4f}")
print(f"{'Up-and-out call (B=130)':25s} {mc_stats(uo_call_payoff)[0]:10.4f} {mc_stats(uo_call_payoff)[1]:10.4f}")
print(f"{'Up-and-in call (B=130)':25s} {mc_stats(ui_call_payoff)[0]:10.4f} {mc_stats(ui_call_payoff)[1]:10.4f}")
print(f"{'  --> Sum (should = vanilla)':25s} {mc_stats(uo_call_payoff)[0] + mc_stats(ui_call_payoff)[0]:10.4f}")
print()
print(f"{'Vanilla put':25s} {mc_stats(vanilla_put_payoff)[0]:10.4f} {mc_stats(vanilla_put_payoff)[1]:10.4f}")
print(f"{'Down-and-out put (B=80)':25s} {mc_stats(do_put_payoff)[0]:10.4f} {mc_stats(do_put_payoff)[1]:10.4f}")
print(f"{'Down-and-in put (B=80)':25s} {mc_stats(di_put_payoff)[0]:10.4f} {mc_stats(di_put_payoff)[1]:10.4f}")
print(f"{'  --> Sum (should = vanilla)':25s} {mc_stats(do_put_payoff)[0] + mc_stats(di_put_payoff)[0]:10.4f}")

In [ ]:
# Barrier sensitivity: how does the up-and-out call price vary with barrier level?
barriers = np.linspace(105, 200, 40)
uo_prices = []
for B in barriers:
    payoff = vanilla_call_payoff * (S_max < B)
    uo_prices.append(np.mean(np.exp(-r*T) * payoff))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(barriers, uo_prices, 'b-', linewidth=2, label='Up-and-out call')
ax.axhline(euro_call, color='r', linestyle='--', label=f'Vanilla call = {euro_call:.4f}')
ax.axvline(K, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Barrier $B$')
ax.set_ylabel('Option price')
ax.set_title(r'Up-and-Out Call Price vs Barrier Level ($B \to \infty$ recovers vanilla)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Lookback Options

A **lookback option** payoff depends on the running maximum or minimum of the asset price over the option's life.

**Floating-strike lookback call**: $\text{payoff} = S_T - S_{\min}$ (buy at the lowest price)

**Floating-strike lookback put**: $\text{payoff} = S_{\max} - S_T$ (sell at the highest price)

**Fixed-strike lookback call**: $\text{payoff} = \max(S_{\max} - K, 0)$

These are the most expensive exotic options because the holder always benefits from hindsight. Closed-form solutions exist but are complex; MC is straightforward.

In [ ]:
# Lookback option pricing
float_call_payoff = S_T - S_min           # floating-strike call: buy at the low
float_put_payoff = S_max - S_T            # floating-strike put: sell at the high
fixed_call_payoff = np.maximum(S_max - K, 0)  # fixed-strike call on the max
fixed_put_payoff = np.maximum(K - S_min, 0)   # fixed-strike put on the min

print("=== Lookback Options ===")
print(f"{'':30s} {'Price':>10s} {'Std Err':>10s} {'vs Vanilla':>12s}")

for name, payoff, vanilla in [
    ("Floating call (S_T - S_min)", float_call_payoff, vanilla_call_payoff),
    ("Floating put (S_max - S_T)", float_put_payoff, vanilla_put_payoff),
    ("Fixed call (max(S_max-K,0))", fixed_call_payoff, vanilla_call_payoff),
    ("Fixed put (max(K-S_min,0))", fixed_put_payoff, vanilla_put_payoff),
]:
    p, e = mc_stats(payoff)
    v, _ = mc_stats(vanilla)
    print(f"{name:30s} {p:10.4f} {e:10.4f} {p/v:12.2f}x")

print("\nLookback options are significantly more expensive than vanilla (2-3x).")

## 4. Price Comparison Across All Option Types

Let's put everything together in a visual comparison.

In [ ]:
# Summary bar chart
labels = [
    'Vanilla\ncall',
    'Asian\ncall',
    'Up-out\ncall\n(B=130)',
    'Fixed LB\ncall',
    'Vanilla\nput',
    'Asian\nput',
    'Down-out\nput\n(B=80)',
    'Fixed LB\nput',
]

prices = [
    mc_stats(vanilla_call_payoff)[0],
    asian_call,
    mc_stats(uo_call_payoff)[0],
    mc_stats(fixed_call_payoff)[0],
    mc_stats(vanilla_put_payoff)[0],
    asian_put,
    mc_stats(do_put_payoff)[0],
    mc_stats(fixed_put_payoff)[0],
]

errors = [
    mc_stats(vanilla_call_payoff)[1],
    asian_call_err,
    mc_stats(uo_call_payoff)[1],
    mc_stats(fixed_call_payoff)[1],
    mc_stats(vanilla_put_payoff)[1],
    asian_put_err,
    mc_stats(do_put_payoff)[1],
    mc_stats(fixed_put_payoff)[1],
]

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2'] * 2

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, prices, yerr=[2*e for e in errors], color=colors, alpha=0.8, capsize=3)
ax.set_ylabel('Option Price')
ax.set_title('Path-Dependent Option Prices Compared')

# Add price labels on bars
for bar, price in zip(bars, prices):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{price:.2f}', ha='center', va='bottom', fontsize=9)

ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Summary

| Option Type | Key Feature | Price vs Vanilla | Why |
|-------------|-------------|:----------------:|-----|
| **Asian** | Averages $S_t$ over time | Cheaper | Averaging reduces effective volatility |
| **Barrier (knock-out)** | Dies if barrier hit | Cheaper | Some paths get knocked out |
| **Barrier (knock-in)** | Lives only if barrier hit | Cheaper | Only some paths activate |
| **Lookback** | Uses $S_{\max}$ or $S_{\min}$ | More expensive | Perfect hindsight always helps |

Key takeaways:
- All path-dependent options are priced with the **same MC engine** — only the payoff computation changes.
- **Barrier parity** holds: knock-in + knock-out = vanilla (verified numerically).
- **Asian options** are ~55-60% of vanilla price due to the variance reduction from averaging.
- **Lookback options** are the most expensive (2-3x vanilla) because the holder benefits from hindsight.
- As the barrier $B \to \infty$ (or $B \to 0$), barrier options converge to vanilla prices.